# 📊 SINITT G2 — Notebook de Análisis
**Equipo G2 · Ruta de Data y Analítica · EAFIT · Grow Data**

Este notebook ejecuta el ciclo completo de análisis en dos capas:

| Capa | ¿Qué hace? | ¿Para qué? |
|---|---|---|
| **EDA** | Medidas de tendencia central y dispersión por base de datos | Entender los datos antes de concluir |
| **KPIs** | Cálculo de los 4 indicadores del dashboard | Responder preguntas de movilidad |

> **Responsable:** Juan Zúñiga Giraldo (DA)  
> **Recibe datos certificados de:** Carol Licet Ospina (DQA)  
> **Entrega resultados a:** Luz Duque (BI Developer)

> **Fuente de datos:** Supabase — tablas cargadas por María Alejandra Valencia (DE)

---
**⚠️ ANTES DE EJECUTAR:** Verifica en la Sección 2 que las credenciales de Supabase
estén guardadas en los Secretos de Colab (ícono de llave en el panel izquierdo).

## 0. Instalación de dependencias
Ejecutar solo una vez al abrir el notebook en Colab.

In [ ]:
!pip install supabase --quiet
print('✅ Dependencias instaladas correctamente')

## 1. Importaciones

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from supabase import create_client
from google.colab import userdata
import warnings
warnings.filterwarnings('ignore')

# Configuración visual global
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_theme(style='whitegrid', palette='muted')

# Paleta de colores del proyecto
COLORES = ['#2563EB', '#0891B2', '#16A34A', '#D97706', '#7C3AED', '#DC2626']

print(f'✅ Librerías listas — pandas {pd.__version__} | numpy {np.__version__}')

## 2. Zona de Configuración
> ⚠️ Las credenciales se leen desde los **Secretos de Colab** — nunca las escribas directamente aquí.
> Para guardarlas: panel izquierdo → ícono de llave → agregar `SUPABASE_URL` y `SUPABASE_KEY`
> con el toggle **'Notebook access'** activado.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CREDENCIALES SUPABASE — leídas desde Secretos de Colab
# ══════════════════════════════════════════════════════════════════════
SUPABASE_URL = userdata.get('SUPABASE_URL')
SUPABASE_KEY = userdata.get('SUPABASE_KEY')

# Verificación rápida
print(f'URL : {SUPABASE_URL[:35]}...' if SUPABASE_URL else '❌ SUPABASE_URL no encontrada en Secretos')
print(f'KEY : {SUPABASE_KEY[:20]}...' if SUPABASE_KEY else '❌ SUPABASE_KEY no encontrada en Secretos')

# ══════════════════════════════════════════════════════════════════════
# NOMBRES DE TABLAS EN SUPABASE
# ══════════════════════════════════════════════════════════════════════
TABLA_BD2 = 'rutas'
TABLA_BD3 = 'paradas'
TABLA_BD5 = 'velocidad_trafico'
TABLA_BD6 = 'intensidad'
TABLA_BD7 = 'incidentes_viales'

# ══════════════════════════════════════════════════════════════════════
# BD5 — velocidad_trafico (2019–2020)
# ══════════════════════════════════════════════════════════════════════
BD5_CORREDOR  = 'nombre_corredor'
BD5_VELOCIDAD = 'velocidad_km_h'
BD5_HORA      = 'hora'
BD5_FECHA     = 'fecha'

# ══════════════════════════════════════════════════════════════════════
# BD6 — intensidad (jul–ago 2020)
# ══════════════════════════════════════════════════════════════════════
BD6_CORREDOR   = 'corredor'
BD6_VELOCIDAD  = 'velocidad_km_h'
BD6_INTENSIDAD = 'intensidad'
BD6_CAT1       = 'categoria_1'
BD6_CAT2       = 'categoria_2'
BD6_CAT3       = 'categoria_3'
BD6_HORA       = 'hora'
BD6_LAT        = 'latitud'
BD6_LON        = 'longitud'

# ══════════════════════════════════════════════════════════════════════
# BD7 — incidentes_viales (2014–2020)
# ══════════════════════════════════════════════════════════════════════
BD7_FECHA    = 'fecha_accidentes'
BD7_CORREDOR = 'comuna'
BD7_GRAVEDAD = 'gravedad_accidente'
BD7_LAT      = 'latitud'
BD7_LON      = 'longitud'

# ══════════════════════════════════════════════════════════════════════
# BD2 — rutas (249 rutas)
# ══════════════════════════════════════════════════════════════════════
BD2_ID_RUTA  = 'id_ruta'
BD2_LONGITUD = 'longitud'

# ══════════════════════════════════════════════════════════════════════
# BD3 — paradas (10,043 paradas)
# ══════════════════════════════════════════════════════════════════════
BD3_ID_RUTA  = 'id_ruta'

# ══════════════════════════════════════════════════════════════════════
# PARÁMETROS ANALÍTICOS
# ══════════════════════════════════════════════════════════════════════
HORAS_PICO            = [6, 7, 8, 9, 17, 18, 19]
HORAS_VALLE           = [1, 2, 3, 4, 5, 22, 23, 0]
PERCENTIL_FLUJO_LIBRE = 85
UMBRAL_IVH_CRITICO    = 3000

print('\n✅ Configuración lista')
print(f'   Horas pico : {HORAS_PICO}')
print(f'   Horas valle: {HORAS_VALLE}')

## 3. Carga de Datos desde Supabase

In [ ]:
# Crear cliente Supabase
supabase = create_client(SUPABASE_URL, SUPABASE_KEY)
print('✅ Cliente Supabase creado correctamente')

In [ ]:
def cargar_tabla(nombre_tabla, nombre_legible):
    """
    Lee una tabla completa de Supabase con paginación automática.
    Supabase limita a 1,000 filas por consulta — esta función
    itera hasta traer todos los registros disponibles.
    """
    try:
        todos = []
        offset = 0
        limite = 1000

        while True:
            respuesta = (
                supabase.table(nombre_tabla)
                .select('*')
                .range(offset, offset + limite - 1)
                .execute()
            )
            batch = respuesta.data
            if not batch:
                break
            todos.extend(batch)
            offset += limite
            if len(batch) < limite:
                break

        df = pd.DataFrame(todos)
        print(f'✅ {nombre_legible}: {df.shape[0]:,} registros | {df.shape[1]} columnas')
        return df

    except Exception as e:
        print(f'❌ {nombre_legible}: error → {e}')
        return None

print('── Cargando tablas desde Supabase ─────────────────────────────────')
df_bd2 = cargar_tabla(TABLA_BD2, 'BD2 Rutas de Transporte (249 rutas)')
df_bd3 = cargar_tabla(TABLA_BD3, 'BD3 Paradas de Transporte (10,043 paradas)')
df_bd5 = cargar_tabla(TABLA_BD5, 'BD5 Velocidad y Tiempo de Viaje (2019-2020)')
df_bd6 = cargar_tabla(TABLA_BD6, 'BD6 Velocidad e Intensidad Vehicular (jul-ago 2020)')
df_bd7 = cargar_tabla(TABLA_BD7, 'BD7 Incidentes Viales Medellín (270,693 registros)')
print('────────────────────────────────────────────────────────────────────')

### 3.1 Verificar columnas reales
> Ejecuta esta celda después de cargar los datos para confirmar que
> los nombres de columna coinciden con los definidos en la Sección 2.

In [ ]:
def mostrar_columnas(df, nombre):
    if df is not None:
        print(f'\n📋 {nombre}')
        print(f'   Columnas: {list(df.columns)}')
        print(f'   Muestra :')
        print(df.head(2).to_string())

mostrar_columnas(df_bd2, 'BD2 — rutas')
mostrar_columnas(df_bd3, 'BD3 — paradas')
mostrar_columnas(df_bd5, 'BD5 — velocidad_trafico')
mostrar_columnas(df_bd6, 'BD6 — intensidad')
mostrar_columnas(df_bd7, 'BD7 — incidentes_viales')

### 3.2 Correcciones de calidad y conversión de tipos
Ajustes necesarios después de leer desde Supabase.

In [ ]:
# ── BD5: Convertir tipos ─────────────────────────────────────────────────
if df_bd5 is not None:
    df_bd5[BD5_FECHA]     = pd.to_datetime(df_bd5[BD5_FECHA], errors='coerce')
    df_bd5[BD5_HORA]      = pd.to_numeric(df_bd5[BD5_HORA], errors='coerce')
    df_bd5[BD5_VELOCIDAD] = pd.to_numeric(df_bd5[BD5_VELOCIDAD], errors='coerce')
    print(f'✅ BD5: tipos convertidos | nulos velocidad: {df_bd5[BD5_VELOCIDAD].isna().sum()}')

# ── BD6: Convertir tipos ─────────────────────────────────────────────────
if df_bd6 is not None:
    df_bd6[BD6_HORA]      = pd.to_numeric(df_bd6[BD6_HORA], errors='coerce')
    df_bd6[BD6_VELOCIDAD] = pd.to_numeric(df_bd6[BD6_VELOCIDAD], errors='coerce')
    df_bd6[BD6_INTENSIDAD]= pd.to_numeric(df_bd6[BD6_INTENSIDAD], errors='coerce')
    print(f'✅ BD6: tipos convertidos | nulos velocidad: {df_bd6[BD6_VELOCIDAD].isna().sum()}')

# ── BD7: Convertir fecha y extraer año/mes ────────────────────────────────
if df_bd7 is not None:
    df_bd7[BD7_FECHA] = pd.to_datetime(df_bd7[BD7_FECHA], errors='coerce')
    df_bd7['anio']    = df_bd7[BD7_FECHA].dt.year
    df_bd7['mes_num'] = df_bd7[BD7_FECHA].dt.month
    print(f'✅ BD7: fecha convertida | nulos fecha: {df_bd7[BD7_FECHA].isna().sum()}')

print('\n✅ Correcciones aplicadas — datos listos para EDA')

## 4. Análisis Exploratorio (EDA)
Antes de calcular cualquier KPI, necesitamos entender la distribución y variabilidad
de cada columna clave.

**Medidas de tendencia central** → dónde se agrupa el fenómeno (media, mediana, moda)

**Medidas de dispersión** → qué tan volátil es (desv. estándar, IQR, percentiles)

El **percentil 85** es especialmente importante: es la velocidad de flujo libre
que usaremos directamente en el cálculo del **KPI 1 — Índice de Congestión Vial**.

In [ ]:
def eda_numerica(serie, nombre_col, nombre_bd):
    """
    Calcula medidas de tendencia central y dispersión.
    El percentil 85 es el insumo directo del KPI 1 (velocidad de flujo libre).
    """
    s = serie.dropna()
    iqr = s.quantile(0.75) - s.quantile(0.25)
    lim_inf = s.quantile(0.25) - 1.5 * iqr
    lim_sup = s.quantile(0.75) + 1.5 * iqr
    outliers = ((s < lim_inf) | (s > lim_sup)).sum()

    stats = {
        'registros_validos' : len(s),
        'nulos'             : serie.isna().sum(),
        'media'             : s.mean(),
        'mediana'           : s.median(),
        'moda'              : s.mode().iloc[0] if len(s.mode()) > 0 else None,
        'desv_estandar'     : s.std(),
        'minimo'            : s.min(),
        'p25'               : s.quantile(0.25),
        'p75'               : s.quantile(0.75),
        'p85'               : s.quantile(0.85),
        'maximo'            : s.max(),
        'iqr'               : iqr,
        'outliers_iqr'      : int(outliers),
    }

    etiquetas = {
        'registros_validos' : 'Registros válidos',
        'nulos'             : 'Valores nulos',
        'media'             : 'Media              ← tendencia central',
        'mediana'           : 'Mediana            ← tendencia central',
        'moda'              : 'Moda               ← tendencia central',
        'desv_estandar'     : 'Desv. Estándar     ← dispersión',
        'minimo'            : 'Mínimo',
        'p25'               : 'Percentil 25',
        'p75'               : 'Percentil 75',
        'p85'               : 'Percentil 85       ← INSUMO KPI 1 (flujo libre)',
        'maximo'            : 'Máximo',
        'iqr'               : 'IQR                ← dispersión',
        'outliers_iqr'      : 'Outliers estimados (método IQR)',
    }

    print(f'\n📊 {nombre_bd} | Columna: {nombre_col}')
    print('─' * 62)
    for k, label in etiquetas.items():
        v = stats[k]
        if isinstance(v, float):
            print(f'  {label:<44}: {v:,.2f}')
        else:
            print(f'  {label:<44}: {v}')
    return stats

print('✅ Función EDA lista')

### 4.1 EDA — BD5: velocidad_trafico (2019–2020)
De aquí extraemos el **percentil 85 en horas valle** = velocidad de flujo libre del **KPI 1**.

In [ ]:
if df_bd5 is not None:
    stats_bd5 = eda_numerica(df_bd5[BD5_VELOCIDAD], BD5_VELOCIDAD, 'BD5 velocidad_trafico')

    print('\n📋 Estadísticos por corredor vial (BD5):')
    resumen_corredor = (
        df_bd5.groupby(BD5_CORREDOR)[BD5_VELOCIDAD]
        .agg(
            Media='mean',
            Mediana='median',
            Desv_Std='std',
            P85_flujo_libre=lambda x: x.quantile(0.85)
        )
        .round(2)
        .sort_values('Media')
    )
    print(resumen_corredor.to_string())
else:
    print('⚠️  BD5 no cargada')

In [ ]:
# Heatmap: velocidad promedio por corredor × hora
if df_bd5 is not None:
    pivot = df_bd5.pivot_table(
        values=BD5_VELOCIDAD,
        index=BD5_CORREDOR,
        columns=BD5_HORA,
        aggfunc='mean'
    )
    fig, ax = plt.subplots(figsize=(16, 6))
    sns.heatmap(
        pivot, cmap='RdYlGn', annot=False, linewidths=0.3,
        cbar_kws={'label': 'Velocidad promedio (km/h)'}, ax=ax
    )
    ax.set_title('EDA BD5 — Velocidad promedio por corredor y hora del día',
                 fontsize=13, pad=12)
    ax.set_xlabel('Hora del día')
    ax.set_ylabel('Corredor vial')
    plt.tight_layout()
    plt.savefig('eda_bd5_heatmap_velocidad.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✅ eda_bd5_heatmap_velocidad.png guardado')

### 4.2 EDA — BD6: intensidad (jul–ago 2020)
Tiene la desagregación por tipo de vehículo que necesita el **KPI 3**.

In [ ]:
if df_bd6 is not None:
    stats_bd6_vel = eda_numerica(df_bd6[BD6_VELOCIDAD], BD6_VELOCIDAD, 'BD6 intensidad')

    if BD6_INTENSIDAD in df_bd6.columns:
        stats_bd6_int = eda_numerica(df_bd6[BD6_INTENSIDAD], BD6_INTENSIDAD, 'BD6 Intensidad total')

    cats = [c for c in [BD6_CAT1, BD6_CAT2, BD6_CAT3] if c in df_bd6.columns]
    if cats:
        print('\n📋 Intensidad promedio por tipo de vehículo (BD6):')
        labels = {BD6_CAT1:'Livianos', BD6_CAT2:'Motos/Ciclos', BD6_CAT3:'Pesados'}
        for c in cats:
            print(f'   {labels.get(c,c):<15}: {df_bd6[c].mean():,.1f} veh/h promedio')
else:
    print('⚠️  BD6 no cargada')

In [ ]:
# Barras apiladas: intensidad por hora y tipo de vehículo
if df_bd6 is not None:
    cats = [c for c in [BD6_CAT1, BD6_CAT2, BD6_CAT3] if c in df_bd6.columns]
    if cats:
        por_hora = df_bd6.groupby(BD6_HORA)[cats].mean()
        labels_leg = ['Livianos', 'Motos/Ciclos', 'Pesados'][:len(cats)]

        fig, ax = plt.subplots(figsize=(13, 5))
        bottom = np.zeros(len(por_hora))
        for i, (cat, label) in enumerate(zip(cats, labels_leg)):
            ax.bar(por_hora.index, por_hora[cat], bottom=bottom,
                   label=label, color=COLORES[i], edgecolor='white', linewidth=0.4)
            bottom += por_hora[cat].values
        ax.axhline(UMBRAL_IVH_CRITICO, color='#DC2626', linestyle='--',
                   linewidth=1.2, label=f'Umbral crítico ({UMBRAL_IVH_CRITICO:,} veh/h)')
        ax.set_title('EDA BD6 — Intensidad vehicular promedio por hora y tipo',
                     fontsize=13, pad=12)
        ax.set_xlabel('Hora del día')
        ax.set_ylabel('Intensidad promedio (veh/h)')
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
        ax.legend()
        plt.tight_layout()
        plt.savefig('eda_bd6_intensidad_horaria.png', dpi=150, bbox_inches='tight')
        plt.show()
        print('✅ eda_bd6_intensidad_horaria.png guardado')

### 4.3 EDA — BD7: incidentes_viales (2014–2020)
270,693 registros. Base del **KPI 2 — Accidentalidad Vial**.

In [ ]:
if df_bd7 is not None:
    if BD7_GRAVEDAD in df_bd7.columns:
        print('📋 Distribución por gravedad de accidente (BD7):')
        dist = df_bd7[BD7_GRAVEDAD].value_counts()
        for grav, cant in dist.items():
            pct = cant / len(df_bd7) * 100
            print(f'   {grav:<30}: {cant:>8,} ({pct:.1f}%)')

    if 'anio' in df_bd7.columns:
        print('\n📋 Incidentes por año:')
        por_anio = df_bd7['anio'].value_counts().sort_index()
        for anio, cant in por_anio.items():
            print(f'   {anio}: {cant:,}')

    if BD7_CORREDOR in df_bd7.columns:
        print('\n📋 Top 10 comunas con más incidentes:')
        top_comunas = df_bd7[BD7_CORREDOR].value_counts().head(10)
        for comuna, cant in top_comunas.items():
            print(f'   {comuna:<25}: {cant:,}')
else:
    print('⚠️  BD7 no cargada')

In [ ]:
# Serie temporal mensual de incidentes
if df_bd7 is not None:
    serie_mensual = (
        df_bd7.groupby(df_bd7[BD7_FECHA].dt.to_period('M'))
        .size().reset_index(name='incidentes')
    )
    serie_mensual[BD7_FECHA] = serie_mensual[BD7_FECHA].dt.to_timestamp()

    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(serie_mensual[BD7_FECHA], serie_mensual['incidentes'],
            color='#DC2626', linewidth=1.2, alpha=0.85)
    ax.fill_between(serie_mensual[BD7_FECHA], serie_mensual['incidentes'],
                    alpha=0.15, color='#DC2626')
    ax.set_title('EDA BD7 — Serie temporal de incidentes viales (2014–2020)',
                 fontsize=13, pad=12)
    ax.set_xlabel('Mes')
    ax.set_ylabel('Incidentes')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
    plt.tight_layout()
    plt.savefig('eda_bd7_serie_temporal.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✅ eda_bd7_serie_temporal.png guardado')

### 4.4 EDA — BD2 + BD3: rutas y paradas
249 rutas y 10,043 paradas. Se cruzan por `id_ruta` para el **KPI 4**.

In [ ]:
if df_bd2 is not None and df_bd3 is not None:
    paradas_x_ruta = df_bd3.groupby(BD3_ID_RUTA).size().reset_index(name='num_paradas')
    print(f'📋 Rutas con paradas registradas: {paradas_x_ruta.shape[0]}')
    eda_numerica(paradas_x_ruta['num_paradas'], 'paradas por ruta', 'BD3')

    if BD2_LONGITUD in df_bd2.columns:
        df_bd2[BD2_LONGITUD] = pd.to_numeric(df_bd2[BD2_LONGITUD], errors='coerce')
        print()
        eda_numerica(df_bd2[BD2_LONGITUD], BD2_LONGITUD, 'BD2 — longitud de rutas (km)')
    else:
        print(f'\n⚠️  Columna "{BD2_LONGITUD}" no encontrada.')
        print(f'   Columnas disponibles: {list(df_bd2.columns)}')
else:
    print('⚠️  BD2 o BD3 no cargadas')

## 5. KPIs del Dashboard
---
### KPI 1 — Índice de Congestión Vial (ICV)
**Fórmula:** `ICV (%) = ((Vel. flujo libre − Vel. observada) / Vel. flujo libre) × 100`

**Vel. flujo libre** = percentil 85 de velocidad en horas valle por corredor.

| ICV | Estado |
|---|---|
| < 20% | Flujo libre 🟢 |
| 20–50% | Congestión moderada 🟡 |
| > 50% | Congestión crítica 🔴 |

**Fuente:** BD5 — velocidad_trafico

In [ ]:
def calcular_icv(df, col_corredor, col_velocidad, col_hora,
                 horas_valle, horas_pico, percentil=85):
    flujo_libre = (
        df[df[col_hora].isin(horas_valle)]
        .groupby(col_corredor)[col_velocidad]
        .quantile(percentil / 100)
        .reset_index()
        .rename(columns={col_velocidad: 'vel_flujo_libre'})
    )
    vel_pico = (
        df[df[col_hora].isin(horas_pico)]
        .groupby(col_corredor)[col_velocidad]
        .mean()
        .reset_index()
        .rename(columns={col_velocidad: 'vel_observada'})
    )
    resultado = flujo_libre.merge(vel_pico, on=col_corredor, how='inner')
    resultado['ICV'] = (
        (resultado['vel_flujo_libre'] - resultado['vel_observada'])
        / resultado['vel_flujo_libre'] * 100
    ).clip(lower=0).round(1)
    resultado['estado'] = pd.cut(
        resultado['ICV'],
        bins=[-1, 20, 50, 100],
        labels=['Flujo libre 🟢', 'Congestión moderada 🟡', 'Congestión crítica 🔴']
    )
    return resultado.sort_values('ICV', ascending=False)

print('✅ Función KPI 1 (ICV) lista')

In [ ]:
if df_bd5 is not None:
    kpi1 = calcular_icv(
        df_bd5, BD5_CORREDOR, BD5_VELOCIDAD, BD5_HORA,
        HORAS_VALLE, HORAS_PICO, PERCENTIL_FLUJO_LIBRE
    )
    print('📊 KPI 1 — ICV por corredor vial:')
    print(kpi1.to_string(index=False))
    print(f'\n🎯 ICV General Medellín     : {kpi1["ICV"].mean():.1f}%')
    print(f'   Corredor más congestionado: {kpi1.iloc[0][BD5_CORREDOR]} ({kpi1.iloc[0]["ICV"]:.1f}%)')
    print(f'   Corredor con mejor flujo  : {kpi1.iloc[-1][BD5_CORREDOR]} ({kpi1.iloc[-1]["ICV"]:.1f}%)')
    n_crit = (kpi1['estado'] == 'Congestión crítica 🔴').sum()
    print(f'   Corredores en estado crítico: {n_crit} de {len(kpi1)}')
else:
    print('⚠️  BD5 no disponible para KPI 1')

In [ ]:
if df_bd5 is not None and 'kpi1' in dir():
    color_map = {
        'Flujo libre 🟢'         : '#16A34A',
        'Congestión moderada 🟡' : '#D97706',
        'Congestión crítica 🔴'  : '#DC2626'
    }
    colores_barra = kpi1['estado'].map(color_map).fillna('#64748B')
    fig, ax = plt.subplots(figsize=(11, max(4, len(kpi1) * 0.55)))
    bars = ax.barh(kpi1[BD5_CORREDOR], kpi1['ICV'],
                   color=colores_barra, edgecolor='white', height=0.65)
    ax.axvline(20, color='#D97706', linestyle='--', linewidth=1, alpha=0.7,
               label='Umbral moderado (20%)')
    ax.axvline(50, color='#DC2626', linestyle='--', linewidth=1, alpha=0.7,
               label='Umbral crítico (50%)')
    for bar, val in zip(bars, kpi1['ICV']):
        ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
                f'{val:.1f}%', va='center', fontsize=9)
    ax.set_title('KPI 1 — Índice de Congestión Vial por corredor', fontsize=13, pad=12)
    ax.set_xlabel('ICV (%)')
    ax.legend(fontsize=9)
    ax.invert_yaxis()
    plt.tight_layout()
    plt.savefig('kpi1_icv_corredores.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✅ kpi1_icv_corredores.png guardado')

### KPI 2 — Accidentalidad Vial por Comuna
**Fuente:** BD7 — incidentes_viales (270,693 registros, 2014–2020)

Agrupado por `comuna`, mes y `gravedad_accidente`.

In [ ]:
def calcular_accidentalidad(df, col_fecha, col_corredor, col_gravedad):
    por_corredor_gravedad = (
        df.groupby([col_corredor, col_gravedad])
        .size().reset_index(name='incidentes')
        .sort_values('incidentes', ascending=False)
    )
    top10 = (
        df.groupby(col_corredor).size()
        .reset_index(name='total')
        .sort_values('total', ascending=False)
        .head(10)
    )
    serie = (
        df.groupby(df[col_fecha].dt.to_period('M')).size()
        .reset_index(name='incidentes')
    )
    serie[col_fecha] = serie[col_fecha].dt.to_timestamp()
    return por_corredor_gravedad, top10, serie

print('✅ Función KPI 2 (Accidentalidad) lista')

In [ ]:
if df_bd7 is not None:
    kpi2_detalle, kpi2_top10, kpi2_serie = calcular_accidentalidad(
        df_bd7, BD7_FECHA, BD7_CORREDOR, BD7_GRAVEDAD
    )
    print('📊 KPI 2 — Top 10 comunas con más incidentes:')
    print(kpi2_top10.to_string(index=False))

    fig, ax = plt.subplots(figsize=(11, 5))
    ax.barh(kpi2_top10[BD7_CORREDOR].astype(str), kpi2_top10['total'],
            color='#DC2626', alpha=0.85, edgecolor='white')
    ax.set_title('KPI 2 — Top 10 comunas por accidentalidad vial (2014–2020)',
                 fontsize=13, pad=12)
    ax.set_xlabel('Total de incidentes')
    ax.invert_yaxis()
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
    plt.tight_layout()
    plt.savefig('kpi2_accidentalidad_top10.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✅ kpi2_accidentalidad_top10.png guardado')
else:
    print('⚠️  BD7 no disponible para KPI 2')

### KPI 3 — Intensidad Vehicular Horaria (IVH)
**Fuente:** BD6 — intensidad (jul–ago 2020)

**Umbral:** IVH > 3,000 veh/h → corredor al límite de capacidad

In [ ]:
def calcular_intensidad_horaria(df, col_corredor, col_hora, col_c1, col_c2, col_c3):
    cats = [c for c in [col_c1, col_c2, col_c3] if c in df.columns]
    detalle = (
        df.groupby([col_corredor, col_hora])[cats]
        .sum().reset_index()
    )
    detalle['IVH_total'] = detalle[cats].sum(axis=1)
    resumen = (
        detalle.groupby(col_corredor)['IVH_total']
        .agg(IVH_promedio='mean', IVH_maximo='max')
        .round(0).sort_values('IVH_maximo', ascending=False)
        .reset_index()
    )
    resumen['estado'] = resumen['IVH_maximo'].apply(
        lambda x: 'Al límite 🔴' if x > UMBRAL_IVH_CRITICO else 'Normal 🟢'
    )
    return detalle, resumen

print('✅ Función KPI 3 (Intensidad Horaria) lista')

In [ ]:
if df_bd6 is not None:
    kpi3_detalle, kpi3_resumen = calcular_intensidad_horaria(
        df_bd6, BD6_CORREDOR, BD6_HORA, BD6_CAT1, BD6_CAT2, BD6_CAT3
    )
    print('📊 KPI 3 — Intensidad vehicular máxima por corredor:')
    print(kpi3_resumen.to_string(index=False))

    corredor_max = kpi3_resumen.iloc[0][BD6_CORREDOR]
    datos_max = kpi3_detalle[kpi3_detalle[BD6_CORREDOR] == corredor_max].sort_values(BD6_HORA)
    cats = [c for c in [BD6_CAT1, BD6_CAT2, BD6_CAT3] if c in datos_max.columns]
    labels_leg = ['Livianos', 'Motos/Ciclos', 'Pesados'][:len(cats)]

    if cats:
        fig, ax = plt.subplots(figsize=(13, 5))
        bottom = np.zeros(len(datos_max))
        for i, (cat, label) in enumerate(zip(cats, labels_leg)):
            ax.bar(datos_max[BD6_HORA], datos_max[cat], bottom=bottom,
                   label=label, color=COLORES[i], edgecolor='white', linewidth=0.3)
            bottom += datos_max[cat].values
        ax.axhline(UMBRAL_IVH_CRITICO, color='#DC2626', linestyle='--',
                   linewidth=1.2, label=f'Umbral ({UMBRAL_IVH_CRITICO:,} veh/h)')
        ax.set_title(f'KPI 3 — Intensidad vehicular horaria: corredor {corredor_max}',
                     fontsize=13, pad=12)
        ax.set_xlabel('Hora del día')
        ax.set_ylabel('Vehículos por hora')
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
        ax.legend()
        plt.tight_layout()
        plt.savefig('kpi3_intensidad_horaria.png', dpi=150, bbox_inches='tight')
        plt.show()
        print('✅ kpi3_intensidad_horaria.png guardado')
else:
    print('⚠️  BD6 no disponible para KPI 3')

### KPI 4 — Eficiencia de la Red de Transporte Público
**Fórmula:** `Densidad (paradas/km) = Número de paradas ÷ Longitud de la ruta (km)`

| Densidad | Cobertura |
|---|---|
| < 2 paradas/km | Baja 🔴 |
| 2–7 paradas/km | Adecuada 🟢 |
| > 7 paradas/km | Sobredotación 🟡 |

**Fuentes:** BD2 — rutas + BD3 — paradas (join por `id_ruta`)

In [ ]:
def calcular_densidad_red(df_rutas, df_paradas, col_id_r, col_long, col_id_p):
    paradas_x_ruta = (
        df_paradas.groupby(col_id_p).size()
        .reset_index(name='num_paradas')
    )
    resultado = (
        df_rutas[[col_id_r, col_long]]
        .merge(paradas_x_ruta, left_on=col_id_r, right_on=col_id_p, how='inner')
    )
    resultado[col_long] = pd.to_numeric(resultado[col_long], errors='coerce')
    resultado['densidad_paradas_km'] = (
        resultado['num_paradas'] / resultado[col_long]
    ).round(2)
    resultado['cobertura'] = pd.cut(
        resultado['densidad_paradas_km'],
        bins=[-1, 2, 7, float('inf')],
        labels=['Baja 🔴', 'Adecuada 🟢', 'Sobredotación 🟡']
    )
    return resultado.sort_values('densidad_paradas_km')

print('✅ Función KPI 4 (Densidad de Red) lista')

In [ ]:
if df_bd2 is not None and df_bd3 is not None:
    kpi4 = calcular_densidad_red(df_bd2, df_bd3, BD2_ID_RUTA, BD2_LONGITUD, BD3_ID_RUTA)
    print('📊 KPI 4 — Resumen densidad de red:')
    eda_numerica(kpi4['densidad_paradas_km'], 'densidad_paradas_km', 'KPI 4')
    print('\nDistribución por categoría:')
    for cat, cnt in kpi4['cobertura'].value_counts().items():
        print(f'   {cat}: {cnt} rutas ({cnt/len(kpi4)*100:.1f}%)')

    fig, ax = plt.subplots(figsize=(11, 4))
    ax.hist(kpi4['densidad_paradas_km'].dropna(), bins=30,
            color='#0891B2', edgecolor='white', alpha=0.85)
    ax.axvline(2, color='#DC2626', linestyle='--', linewidth=1.2, label='Umbral bajo (2)')
    ax.axvline(7, color='#D97706', linestyle='--', linewidth=1.2, label='Sobredotación (7)')
    ax.set_title('KPI 4 — Distribución densidad de paradas por ruta', fontsize=13, pad=12)
    ax.set_xlabel('Paradas por km')
    ax.set_ylabel('Número de rutas')
    ax.legend()
    plt.tight_layout()
    plt.savefig('kpi4_densidad_red.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✅ kpi4_densidad_red.png guardado')
else:
    print('⚠️  BD2 o BD3 no disponibles para KPI 4')

## 6. Resumen Ejecutivo
Consolidación de los 4 KPIs para entregar a Luz (BI Developer).

In [ ]:
print('=' * 62)
print('SINITT G2 — RESUMEN EJECUTIVO DE KPIs')
print('Equipo G2 · Grow Data · EAFIT · Medellín')
print('=' * 62)

if 'kpi1' in dir():
    print(f'\nKPI 1 — Índice de Congestión Vial')
    print(f'  ICV general Medellín        : {kpi1["ICV"].mean():.1f}%')
    print(f'  Corredor más congestionado  : {kpi1.iloc[0][BD5_CORREDOR]} ({kpi1.iloc[0]["ICV"]:.1f}%)')
    print(f'  Corredores estado crítico   : {(kpi1["estado"]=="Congestión crítica 🔴").sum()} de {len(kpi1)}')

if 'kpi2_top10' in dir():
    print(f'\nKPI 2 — Accidentalidad Vial')
    print(f'  Total incidentes 2014-2020  : {len(df_bd7):,}')
    print(f'  Comuna más peligrosa        : {kpi2_top10.iloc[0][BD7_CORREDOR]} ({kpi2_top10.iloc[0]["total"]:,})')

if 'kpi3_resumen' in dir():
    print(f'\nKPI 3 — Intensidad Vehicular Horaria')
    print(f'  Corredor más cargado        : {kpi3_resumen.iloc[0][BD6_CORREDOR]}')
    print(f'  IVH máximo                  : {kpi3_resumen.iloc[0]["IVH_maximo"]:,.0f} veh/h')
    print(f'  Corredores al límite        : {(kpi3_resumen["estado"]=="Al límite 🔴").sum()} de {len(kpi3_resumen)}')

if 'kpi4' in dir():
    baja = (kpi4['cobertura'] == 'Baja 🔴').sum()
    print(f'\nKPI 4 — Densidad de Red de Transporte')
    print(f'  Rutas analizadas            : {len(kpi4)}')
    print(f'  Densidad promedio           : {kpi4["densidad_paradas_km"].mean():.2f} paradas/km')
    print(f'  Rutas con cobertura baja    : {baja} ({baja/len(kpi4)*100:.1f}%)')

print('\n' + '=' * 62)

## 7. Exportación de Resultados
Los CSV generados aquí son los insumos que Luz (BI Developer) usa en Lovable/Bolt.
También se cargan de vuelta en Supabase en tablas de resultados.

In [ ]:
import os
os.makedirs('resultados_kpis', exist_ok=True)

exportados = []

if 'kpi1' in dir():
    kpi1.to_csv('resultados_kpis/kpi1_icv_corredores.csv', index=False)
    exportados.append('kpi1_icv_corredores.csv')

if 'kpi2_top10' in dir():
    kpi2_top10.to_csv('resultados_kpis/kpi2_top10_comunas.csv', index=False)
    kpi2_detalle.to_csv('resultados_kpis/kpi2_detalle_comuna_gravedad.csv', index=False)
    exportados.extend(['kpi2_top10_comunas.csv', 'kpi2_detalle_comuna_gravedad.csv'])

if 'kpi3_resumen' in dir():
    kpi3_resumen.to_csv('resultados_kpis/kpi3_intensidad_horaria.csv', index=False)
    exportados.append('kpi3_intensidad_horaria.csv')

if 'kpi4' in dir():
    kpi4.to_csv('resultados_kpis/kpi4_densidad_red.csv', index=False)
    exportados.append('kpi4_densidad_red.csv')

print('📁 Archivos exportados en resultados_kpis/:')
for f in exportados:
    print(f'   ✅ {f}')
print('\n→ Entregar esta carpeta a Luz (BI Developer) para Lovable/Bolt')

In [ ]:
# ── Cargar resultados en Supabase ─────────────────────────────────────────
# Los resultados de los KPIs se guardan en tablas separadas
# para que Lovable los lea directamente sin recalcular

def cargar_resultado_supabase(df, nombre_tabla, descripcion):
    try:
        # Convertir columnas categóricas a string para serialización JSON
        df_export = df.copy()
        for col in df_export.select_dtypes(include='category').columns:
            df_export[col] = df_export[col].astype(str)
        # Reemplazar NaN por None para compatibilidad con JSON
        registros = df_export.where(pd.notnull(df_export), None).to_dict('records')
        supabase.table(nombre_tabla).upsert(registros).execute()
        print(f'✅ {descripcion}: {len(registros):,} registros cargados en "{nombre_tabla}"')
    except Exception as e:
        print(f'❌ {descripcion}: error → {e}')

print('── Cargando resultados KPIs en Supabase ───────────────────────────')
if 'kpi1' in dir():
    cargar_resultado_supabase(kpi1, 'resultado_kpi1_icv', 'KPI 1 — ICV')

if 'kpi2_top10' in dir():
    cargar_resultado_supabase(kpi2_top10, 'resultado_kpi2_accidentalidad', 'KPI 2 — Accidentalidad')

if 'kpi3_resumen' in dir():
    cargar_resultado_supabase(kpi3_resumen, 'resultado_kpi3_intensidad', 'KPI 3 — Intensidad')

if 'kpi4' in dir():
    cargar_resultado_supabase(kpi4, 'resultado_kpi4_densidad', 'KPI 4 — Densidad de Red')

print('────────────────────────────────────────────────────────────────────')
print('✅ Resultados disponibles en Supabase para Lovable/Bolt')

---
**SINITT G2 · Grupo No. 2 · EAFIT · Entrega final: 20 mayo 2026 · Socialización: 12 junio 2026**

*Responsable del análisis: Juan Zúñiga Giraldo (DA)*  
*Datos certificados por: Carol Licet Ospina (DQA)*  
*Datos cargados en Supabase por: María Alejandra Valencia (DE)*  
*Resultados entregados a: Luz Duque (BI Developer)*  
*Empresa aliada: Grow Data*